In [ ]:
import pandas as pd
data = pd.read_csv("Marvel_Comics.csv")
data

,comic_name,active_years,issue_title,publish_date,issue_description,penciler,writer,cover_artist,Imprint,Format,Rating,Price
0,A Year of Marvels: April Infinite Comic (2016),(2016),A Year of Marvels: April Infinite Comic (2016) #1,"April 01, 2016",The Infinite Comic that will have everyone tal...,Yves Bigerel,Yves Bigerel,Jamal Campbell,Marvel Universe,Infinite Comic,Rated T+,Free
1,A Year of Marvels: August Infinite Comic (2016),(2016),A Year of Marvels: August Infinite Comic (2016...,"August 10, 2016","It’s August, and Nick Fury is just in time to ...",Jamal Campbell,"Chris Sims, Chad Bowers",NaN,Marvel Universe,Infinite Comic,NaN,Free
2,A Year of Marvels: February Infinite Comic (2016),(2016),A Year of Marvels: February Infinite Comic (20...,"February 10, 2016",Join us in a brand new Marvel comics adventure...,"Danilo S. Beyruth, M Mast",Ryan North,NaN,Marvel Universe,Infinite Comic,Rated T+,Free
3,A Year of Marvels: July Infinite Comic (2016),(2016),A Year of Marvels: July Infinite Comic (2016) #1,"June 29, 2016",Celebrating the Fourth of July is complicated ...,Juanan Ramirez,Chuck Wendig,Jamal Campbell,Marvel Universe,Infinite Comic,NaN,Free
4,A Year of Marvels: June Infinite Comic (2016),(2016),A Year of Marvels: June Infinite Comic (2016) #1,"June 15, 2016",Sam Alexander’s finding it hard to cope with t...,Diego Olortegui,Paul Allor,Jamal Campbell,Marvel Universe,Infinite Comic,NaN,Free
...,...,...,...,...,...,...,...,...,...,...,...,...
34987,Ziggy Pig - Silly Seal Comics (2019),(2019),Ziggy Pig - Silly Seal Comics (2019) #1,"March 06, 2019",NOT SO FUNNY WHEN IT HAPPENS TO YOU? Once they...,Jacob Chabot,"John Cerilli, Frank Tieri",Nic Klein,Marvel Universe,Comic,Parental Advisory,$3.99
34988,Zombie (2006),(2006),Zombie (2006) #4,"December 20, 2006",With a thousand zombies in front of him and tw...,Kyle Hotz,Mike Raicht,Kyle Hotz,MAX,Comic,EXPLICIT CONTENT,$3.99
34989,Zombie (2006),(2006),Zombie (2006) #3,"November 29, 2006",The hordes of zombies gathered outside the hig...,Kyle Hotz,Mike Raicht,Kyle Hotz,MAX,Comic,EXPLICIT CONTENT,$3.99
34990,Zombie (2006),(2006),Zombie (2006) #2,"October 25, 2006","For Simon Garth, it's come down to two very ba...",Kyle Hotz,Mike Raicht,Kyle Hotz,MAX,Comic,EXPLICIT CONTENT,$3.99


In [3]:
import re
from collections import Counter

def extract_capitalized_words(text):
    if not isinstance(text, str):
        return []
    return re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', text)

combined_text = data['comic_name'].str.cat(data['issue_title'], sep=' ').str.cat(data['issue_description'], sep=' ')
                                                                                  
capitalized_words = combined_text.apply(extract_capitalized_words).explode().dropna()

word_counts = Counter(capitalized_words)

most_common_words = word_counts.most_common(10)
most_common_words

[('Man', 8861),
 ('Men', 7340),
 ('The', 6916),
 ('Spider', 5684),
 ('But', 4581),
 ('And', 4079),
 ('Avengers', 3463),
 ('Marvel', 2859),
 ('Captain America', 2849),
 ('Wolverine', 2629)]

In [4]:
def refined_extract(text):
    if not isinstance(text, str):
        return []
    return re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', text)

refined_words = combined_text.apply(refined_extract).explode().dropna()
refined_words_counts = Counter(refined_words)

exclude_words =  {"None", "The", "But", "And", "A", "Or", "For", "With", "As", "In", "To", "Of", "Is", "On"}

top_refined_words = [(word, count) for word, count in refined_words_counts.most_common() if word not in exclude_words]

top_10_refined = top_refined_words[:10]
top_10_refined

[('Man', 8861),
 ('Men', 7340),
 ('Spider', 5684),
 ('Avengers', 3463),
 ('Marvel', 2859),
 ('Captain America', 2849),
 ('Wolverine', 2629),
 ('It', 2536),
 ('Thor', 2531),
 ('Iron Man', 2349)]

In [5]:
writers_list = data['writer'].str.split(',|&').explode().str.strip()

writer_data = data.loc[writers_list.index, ['comic_name', 'active_years']]
writer_data['writer'] = writers_list.values

grouped_writers = writer_data.groupby('writer').agg({
    'comic_name': ['count', lambda x: x.str.extractall(r'\b([A-Z][a-z]+(?:[-\s]+[A-Z][a-z]+)*)\b').value_counts().index[:5].tolist()],
    'active_years' : 'unique'
}).sort_values(('comic_name', 'count'), ascending=False)

grouped_writers.columns = ['Number of Comics Written', 'Top 5 Associated Superheroes', 'Years Written']

top_writers = grouped_writers.head(10)
top_writers

,Number of Comics Written,Top 5 Associated Superheroes,Years Written
writer,,,
Stan Lee,980,"[(Tales,), (Fantastic Four,), (Strange Tales,)...","[(1970 - 1975), (2006), (2007), (1941 - 1947),..."
Brian Michael Bendis,906,"[(Ultimate Spider-Man,), (New Avengers,), (Men...","[(2013), (2001 - 2003), (2012 - 2015), (1999 -..."
Chris Claremont,823,"[(Men,), (Uncanny,), (New Mutants,), (Treme,),...","[(1983 - 1994), (1967 - 1994), (2008 - 2009), ..."
Peter David,711,"[(Factor,), (Incredible Hulk,), (Dark Tower,),...","[(2012 - 2014), (2019), (2014 - 2015), (1999 -..."
Roy Thomas,586,"[(Marvel Illustrated,), (Doctor Strange,), (Me...","[(1970 - 1976), (1970), (1963 - 1996), (1998 -..."
Tom Defalco,510,"[(Fantastic Four,), (The Amazing Spider-Man,),...","[(1998 - 1999), (2006 - 2009), (1964 - 2018), ..."
Fabian Nicieza,419,"[(Cable,), (Deadpool,), (Force,), (Men,), (Thu...","[(1991 - 1992), (2015), (1983 - 1994), (1963 -..."
Jason Aaron,399,"[(Wolverine,), (Men,), (Thor,), (Star Wars,), ...","[(2019), (2013 - 2015), (2010 - 2011), (2010),..."
Mark Waid,372,"[(Daredevil,), (Captain America,), (Avengers,)...","[(2013), (2015 - 2016), (1999 - 2013), (2018),..."


In [16]:
# top_writers = top_writers.drop("None")

def format_years(years_list):
    years = sorted([int(year) for sublist in [re.findall(r'(\d+)', year_interval) for year_interval in years_list] for year in sublist])
    
    if not years:
        return ""

    groups = []
    current_group = [years[0]]
    for y in years[1:]:
        if y - current_group[-1] == 1:
            current_group.append(y)
        else:
            groups.append(current_group)
            currten_group = [y]
    groups.append(currten_group)

    formatted_years = [f"{group[0]}-{group[-1]}" if len(group) > 1 else str(group[0]) for group in groups]
    return ", ".join(formatted_years)

top_writers['Years Written'] = top_writers['Years Written'].apply(format_years)
top_writers

C:\Users\2023\AppData\Local\Temp\ipykernel_3340\405004923.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_writers['Years Written'] = top_writers['Years Written'].apply(format_years)


,Number of Comics Written,Top 5 Associated Superheroes,Years Written
writer,,,
Stan Lee,980,"[(Tales,), (Fantastic Four,), (Strange Tales,)...","0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."
Brian Michael Bendis,906,"[(Ultimate Spider-Man,), (New Avengers,), (Men...","0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."
Chris Claremont,823,"[(Men,), (Uncanny,), (New Mutants,), (Treme,),...","0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."
Peter David,711,"[(Factor,), (Incredible Hulk,), (Dark Tower,),...","0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."
Roy Thomas,586,"[(Marvel Illustrated,), (Doctor Strange,), (Me...","0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."
Tom Defalco,510,"[(Fantastic Four,), (The Amazing Spider-Man,),...","0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."
Fabian Nicieza,419,"[(Cable,), (Deadpool,), (Force,), (Men,), (Thu...","0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."
Jason Aaron,399,"[(Wolverine,), (Men,), (Thor,), (Star Wars,), ...","0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."
Mark Waid,372,"[(Daredevil,), (Captain America,), (Avengers,)...","0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."


In [18]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

words_list = data['issue_description'].str.split().explode().str.lower()

filtered_words = words_list[~words_list.isin(ENGLISH_STOP_WORDS) & (words_list != 'none')]

words_counts = Counter(filtered_words)


top_20_words = words_counts.most_common(20)
top_20_words

[('new', 6737),
 (nan, 4597),
 ('-', 4480),
 ('marvel', 4083),
 ("it's", 3264),
 ('man', 2969),
 ('x-men', 2606),
 ('spider-man', 2508),
 ('avengers', 2405),
 ('captain', 2220),
 ('world', 2136),
 ('team', 2055),
 ('iron', 1996),
 ('battle', 1920),
 ('time', 1910),
 ('war', 1823),
 ('just', 1802),
 ('save', 1795),
 ('black', 1786),
 ('...$2.99', 1680)]